In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [4]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [10]:
from sklearn.metrics import classification_report
ths = [15, 16, 17, 18, 19, 19.5, 20, 21]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 15 hidden = 0


,0,1,2,3,4,5,-1
0,0,130365,0,0,0,42480,29383
1,0,124956,16,1,130,1400,2033
2,0,1,82231,0,0,0,73
3,0,0,3,60943,0,0,4
4,0,39,0,0,45671,22,58
5,0,4,0,0,8,134,1
-1,0,0,0,0,0,0,0


th: 15 hidden: 0 acc:0.6634061343652156 recall:0.14529639812488873 precision:0.9312563387423936 f1:0.25137308580716916
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.93      0.15      0.25    202228
           1       0.49      0.97      0.65    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      1.00      1.00     45790
           5       0.00      0.91      0.01       147

    accuracy                           0.66    519956
   macro avg       0.74      0.84      0.65    519956
weighted avg       0.85      0.66      0.62    519956

[0, 1, 2, 3, 4, 5, -1]
th = 16 hidden = 0


,0,1,2,3,4,5,-1
0,0,129982,0,0,0,41028,31218
1,0,123904,14,1,125,1381,3111
2,0,1,82184,0,0,0,120
3,0,0,3,60943,0,0,4
4,0,39,0,0,45650,22,79
5,0,4,0,0,8,132,3
-1,0,0,0,0,0,0,0


th: 16 hidden: 0 acc:0.6647274000107701 recall:0.15437031469430543 precision:0.9039525119444042 f1:0.2637067447194029
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.90      0.15      0.26    202228
           1       0.49      0.96      0.65    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      1.00      1.00     45790
           5       0.00      0.90      0.01       147

    accuracy                           0.66    519956
   macro avg       0.73      0.84      0.65    519956
weighted avg       0.84      0.66      0.63    519956

[0, 1, 2, 3, 4, 5, -1]
th = 17 hidden = 0


...

th: 19.5 hidden: 5 acc:0.9915704405757411 recall:0.16326530612244897 precision:0.0056022408963585435 f1:0.010832769126607989
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.01      0.16      0.01       147
           0       1.00      1.00      1.00    202228
           1       1.00      0.97      0.99    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      0.99     45790

    accuracy                           0.99    519956
   macro avg       0.83      0.85      0.83    519956
weighted avg       1.00      0.99      0.99    519956

[0, 1, 2, 3, 4, 5, -1]
th = 20 hidden = 5


,0,1,2,3,4,5,-1
0,201737,22,0,0,0,0,469
1,132,124477,15,0,84,0,3828
2,0,1,81920,0,0,0,384
3,0,0,3,60876,0,0,71
4,1,47,0,0,45349,0,393
5,0,112,0,0,9,0,26
-1,0,0,0,0,0,0,0


th: 20 hidden: 5 acc:0.9898722199570733 recall:0.17687074829931973 precision:0.0050280409978727516 f1:0.009778112072207596
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.01      0.18      0.01       147
           0       1.00      1.00      1.00    202228
           1       1.00      0.97      0.98    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      0.99     45790

    accuracy                           0.99    519956
   macro avg       0.83      0.85      0.83    519956
weighted avg       1.00      0.99      0.99    519956

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 5


,0,1,2,3,4,5,-1
0,201685,12,0,0,0,0,531
1,126,121436,14,0,61,0,6899
2,0,1,81642,0,0,0,662
3,0,0,3,60876,0,0,71
4,1,47,0,0,44966,0,776
5,0,80,0,0,7,0,60
-1,0,0,0,0,0,0,0


th: 21 hidden: 5 acc:0.9826408388402096 recall:0.40816326530612246 precision:0.00666740748972108 f1:0.013120489831620381
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.01      0.41      0.01       147
           0       1.00      1.00      1.00    202228
           1       1.00      0.94      0.97    128536
           2       1.00      0.99      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.98      0.99     45790

    accuracy                           0.98    519956
   macro avg       0.83      0.89      0.83    519956
weighted avg       1.00      0.98      0.99    519956



# Média absolute threshold

In [11]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 15: 0.3133812217268583
Média f1 absolute th 16: 0.3326080441908207
Média f1 absolute th 17: 0.4628026008354923
Média f1 absolute th 18: 0.6672472629766146
Média f1 absolute th 19: 0.6738994448919152
Média f1 absolute th 19.5: 0.6854770782912036
Média f1 absolute th 20: 0.6787262732239485
Média f1 absolute th 21: 0.6663588048793411
